In [1]:
import torch
import torch.nn as nn
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import pinn_torch as pinn

In [4]:
#rbf layer as input to the neural network
x_max = 4.0
y_max = 1.0
domain_sizes = [x_max,y_max]
num_centers = 11

#physics_head = pinn.pinn.PhysicsHead(num_spatial=1)
rbf_layer = pinn.pinn.RadialBasisFunction(num_centers=11,domain_sizes=domain_sizes)

num_inputs = rbf_layer.rbf_centers.shape[1]
nodes_per_layer = 32
num_layers = 8
resnn = pinn.pinn.ResidualFullyConnected(num_inputs,num_layers,nodes_per_layer)

physics_layer = pinn.pinn.PhysicsLayer(num_inputs=nodes_per_layer,num_linear_outputs=2)



In [5]:
net = pinn.pinn.PhysicsHeadSteady(
    layers=(rbf_layer,resnn,physics_layer),
    num_spatial=2
)

#net.c2 = nn.Parameter(torch.tensor(np.array([1.0])).float())

mse_cost_function = torch.nn.MSELoss() # Mean squared error
optimizer = torch.optim.Adam(net.parameters())

In [6]:
net.to(device)

PhysicsHeadSteady(
  (stack): Sequential(
    (0): RadialBasisFunction()
    (1): ResidualFullyConnected(
      (resnn): Sequential(
        (0): Linear(in_features=121, out_features=32, bias=True)
        (1): Tanh()
        (2): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
        (3): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
        (4): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
        (5): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
        (6): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
        (7): LinearSkip(
          (linear): Linear(in_features=32, out_features=32, bias=True)
          (act): Tanh()
        )
  

In [7]:
output_names = [
    #'p',
    'u','v',#'w'
]

pde_names = [
    'fp','fu','fv'
]

X = torch.tensor([[0.0],[0.5],[1.0]],requires_grad=True).to(device)
Y = torch.tensor([[0.01],[0.5],[1.0]],requires_grad=True).to(device)
spatial = [X,Y]
outputs = net(spatial)
U = outputs
outputs

tensor([[ 0.0061,  0.2193],
        [ 0.0396,  0.0312],
        [-0.1300,  0.1607]], device='cuda:0', grad_fn=<CatBackward0>)

In [8]:
outputs_dict = {name: outputs[:,i:i+1] for i, name in enumerate(output_names)}
outputs_dict

{'u': tensor([[ 0.0061],
         [ 0.0396],
         [-0.1300]], device='cuda:0', grad_fn=<SliceBackward0>),
 'v': tensor([[0.2193],
         [0.0312],
         [0.1607]], device='cuda:0', grad_fn=<SliceBackward0>)}

In [9]:
outputs_list = [outputs_dict[name] for name in output_names]
u_difs = pinn.differentiation.del_scalar(outputs_dict['u'],spatial,)
v_difs = pinn.differentiation.del_scalar(outputs_dict['v'],spatial,)
u_sec = pinn.differentiation.div_vector(u_difs,spatial,)
v_sec = pinn.differentiation.div_vector(v_difs,spatial,)
ux,uy = u_difs
#uxx,uyy = u_sec
#vxx,vyy = v_sec

In [10]:
#u_difs[1],u_difs[1]/spatial[1]

In [11]:
difs = [
    pinn.differentiation.del_scalar(comp,spatial,) 
    #for comp in outputs_list
    for name,comp in outputs_dict.items()
]
difs

[(tensor([[0.0593],
          [0.0721],
          [0.0454]], device='cuda:0', grad_fn=<SliceBackward0>),
  tensor([[ 0.1659],
          [-0.2385],
          [-0.4075]], device='cuda:0', grad_fn=<SliceBackward0>)),
 (tensor([[-0.1050],
          [-0.1676],
          [-0.0840]], device='cuda:0', grad_fn=<SliceBackward0>),
  tensor([[-0.2352],
          [ 0.0463],
          [ 0.4234]], device='cuda:0', grad_fn=<SliceBackward0>))]

In [12]:
divs = [
    pinn.differentiation.div_vector(dif,spatial,) for dif in difs
]
divs

[tensor([[-0.3770],
         [-1.0043],
         [ 0.9048]], device='cuda:0', grad_fn=<SumBackward1>),
 tensor([[-1.0486],
         [ 1.7728],
         [-1.4269]], device='cuda:0', grad_fn=<SumBackward1>)]

In [13]:
outputs_list

[tensor([[ 0.0061],
         [ 0.0396],
         [-0.1300]], device='cuda:0', grad_fn=<SliceBackward0>),
 tensor([[0.2193],
         [0.0312],
         [0.1607]], device='cuda:0', grad_fn=<SliceBackward0>)]

In [14]:
outputs_dict['u']*ux,outputs_dict['v']*uy

(tensor([[ 0.0004],
         [ 0.0029],
         [-0.0059]], device='cuda:0', grad_fn=<MulBackward0>),
 tensor([[ 0.0364],
         [-0.0075],
         [-0.0655]], device='cuda:0', grad_fn=<MulBackward0>))

In [15]:
outputs_dict['u']*ux+outputs_dict['v']*uy

tensor([[ 0.0367],
        [-0.0046],
        [-0.0714]], device='cuda:0', grad_fn=<AddBackward0>)

In [16]:
[[val,der] for (name,val),der in zip(outputs_dict.items(),u_difs)]

[[tensor([[ 0.0061],
          [ 0.0396],
          [-0.1300]], device='cuda:0', grad_fn=<SliceBackward0>),
  tensor([[0.0593],
          [0.0721],
          [0.0454]], device='cuda:0', grad_fn=<SliceBackward0>)],
 [tensor([[0.2193],
          [0.0312],
          [0.1607]], device='cuda:0', grad_fn=<SliceBackward0>),
  tensor([[ 0.1659],
          [-0.2385],
          [-0.4075]], device='cuda:0', grad_fn=<SliceBackward0>)]]

In [17]:
[val*der for (name,val),der in zip(outputs_dict.items(),u_difs)]

[tensor([[ 0.0004],
         [ 0.0029],
         [-0.0059]], device='cuda:0', grad_fn=<MulBackward0>),
 tensor([[ 0.0364],
         [-0.0075],
         [-0.0655]], device='cuda:0', grad_fn=<MulBackward0>)]

In [18]:
(
    sum([val*der for val,der in zip(outputs_list,u_difs)]),
    sum([val*der for val,der in zip(outputs_list,v_difs)]),
)

(tensor([[ 0.0367],
         [-0.0046],
         [-0.0714]], device='cuda:0', grad_fn=<AddBackward0>),
 tensor([[-0.0522],
         [-0.0052],
         [ 0.0790]], device='cuda:0', grad_fn=<AddBackward0>))

In [19]:
[
    sum([val*der for (name,val),der in zip(outputs_dict.items(),dif)])
    for dif in difs
]

[tensor([[ 0.0367],
         [-0.0046],
         [-0.0714]], device='cuda:0', grad_fn=<AddBackward0>),
 tensor([[-0.0522],
         [-0.0052],
         [ 0.0790]], device='cuda:0', grad_fn=<AddBackward0>)]

In [20]:
nu = 1e-5
rho = 1
dpdx = 1

gradp = [dpdx,0]
Porho = [dp/rho for dp in gradp]

U = net(spatial)
cont_eq = pinn.differentiation.div_vector(U,spatial)

U_dict = {name: U[:,i:i+1] for i, name in enumerate(output_names)}

U_diffs = [pinn.differentiation.del_scalar(comp,spatial,) for name,comp in U_dict.items()]
U_divgs = [pinn.differentiation.div_vector(dif,spatial,) for dif in U_diffs]

U_convs = [
    sum([comp*dif_comp for (name,comp),dif_comp in zip(U_dict.items(),diff)]) for diff in U_diffs
]

ns_eq = [
    (conv + porho - nu*divg )
    for conv,porho,divg in zip(U_convs,Porho,U_divgs)
]

pde_list = [cont_eq]+ns_eq
pde_out = {name:pde for name,pde in zip(pde_names,pde_list)}

In [21]:
U_convs

[tensor([[ 0.0367],
         [-0.0046],
         [-0.0714]], device='cuda:0', grad_fn=<AddBackward0>),
 tensor([[-0.0522],
         [-0.0052],
         [ 0.0790]], device='cuda:0', grad_fn=<AddBackward0>)]

In [22]:
[cont_eq]+ns_eq

[tensor([[-0.0457],
         [-0.1923],
         [ 0.0000]], device='cuda:0', grad_fn=<SumBackward1>),
 tensor([[1.0368],
         [0.9954],
         [0.9286]], device='cuda:0', grad_fn=<SubBackward0>),
 tensor([[-0.0522],
         [-0.0052],
         [ 0.0790]], device='cuda:0', grad_fn=<SubBackward0>)]

In [23]:
class Flow:
    def __init__(self,nu=1e-5,rho=1,dpdx=1):
        self.nu = nu
        self.rho = rho
        self.gradp = [dpdx,0]
        self.Porho = [dp/rho for dp in self.gradp]
        
        self.output_names = [
            #'p',
            'u','v',#'w'
        ]
        return
    
    def f(self,model,spatial):
        U = model(spatial)
        
        cont_eq = pinn.differentiation.div_vector(U,spatial)
        
        U_dict = {name: U[:,i:i+1] for i, name in enumerate(self.output_names)}
        
        #velocity first partials and divergence
        U_diffs = [pinn.differentiation.del_scalar(comp,spatial,) for name,comp in U_dict.items()]
        U_divgs = [pinn.differentiation.div_vector(dif,spatial,) for dif in U_diffs]
        
        #convective derivative u dot del(u)
        U_convs = [
            sum(comp*dif_comp) for (name,comp),dif_comp in zip(U_dict.items(),diff)
            for diff in U_diffs
        ]
        
        #conv + P/rho - nu*divg
        ns_eq = [
            (conv + porho - self.nu*divg )
            for conv,porho,divg in zip(U_convs,self.Porho,U_divgs)
        ]
        